<a href="https://colab.research.google.com/github/XiaomanLu/E3SM/blob/master/Planet_LAI_10yrs_constrained_NEW.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Initialization
!pip install earthengine-api --quiet
!pip install geemap --quiet

# Authenticate
import ee
import geemap
ee.Authenticate()

# Initialize the EE API
ee.Initialize(project='quick-heaven-445214-f5')

In [ ]:
# @title Configuration
# We use geemap for Earth Engine handling and rasterio for the local GeoTIFF
!pip install geemap rasterio -q

import rasterio
from rasterio.warp import reproject, Resampling
import numpy as np
import pandas as pd
from rasterio.enums import Resampling

def site_location(site):
    if site in ['z6_SEEED', 'SEEED']:
        site_lat, site_lon = 35.9716431, -83.9047674
    elif site in ['z6-20662-Melshouse', 'VictorAshe-2']:
        site_lat, site_lon = 35.9836745, -83.9928031
    elif site in ['z6-Chilhowee', 'WestView']:
        site_lat, site_lon = 35.9660326, -83.9612832
    elif site in ['z6-Cumberland', 'Cumberland']:
        site_lat, site_lon = 35.9886325, -84.0155383
    elif site in ['z6-Victor_Ashe', 'VictorAshe']:
        site_lat, site_lon = 35.9829284, -83.9926842
    elif site in ['z6-WestHills', 'WestHills']:
        site_lat, site_lon = 35.9316965, -84.0445624
    return(site_lat, site_lon)


# ================= CONFIGURATION =================
Site = 'SEEED'        #options: ['VictorAshe', 'SEEED', 'Cumberland', 'WestHills']
BaseVI = 'NDVI'        #options: ['NDVI', 'EVI']; methods to calculate LAI
BaseStats = 'Median'    #options: ['Median', 'Mean']; methods to aggregate monthly LAI
# choose NDVI+median combination for all sites, due to best pattern of seasonal cycle.

lat, lon = site_location(Site)
img_lc = ee.Image(f'projects/quick-heaven-445214-f5/assets/New_knoxLC_5class_WGS84_100m_{Site}') # 5m Land Cover Tiff; ~2m after reproject

# Define ROI (50m radius circle)
point = ee.Geometry.Point([lon, lat])
roi = point.buffer(50).bounds() # 50m radius buffer square (buffer(50) creates a circle, then .bounds() turns that circle into a square-a bounding box).

# Set the time range
start_date = '2015-01-01'
end_date = '2024-12-31'
bad_date = '2023-07-28' #very low LAI because of smoke or cloud (high AOT or SCL)

# Class Mapping (Value in Tiff : Class Name)
class_map = {
    11: 'Water',
    24: 'Impervious',
    31: 'Barren',
    43: 'Tree',
    71: 'Grass'
}
lc_palette = {
    11: 'blue',
    24: 'pink',
    31: 'gray',
    43: 'green',
    71: 'yellow'
}

# Base map
Map = geemap.Map(center=roi.centroid(1).coordinates().getInfo()[::-1], zoom=14)
Map.add_basemap('SATELLITE')
# =================================================



In [ ]:
# @title Planet LAI
def calculate_lai(image):
    """
    Calculates empirical LAI using a standard NDVI-based formula.
    Formula: LAI = 0.57 * exp(2.33 * NDVI)
    Note: For higher accuracy, consider using the SNAP Biophysical Processor,
    but this is the standard lightweight method for scripts.
    """

    if BaseVI == 'NDVI':
      ndvi = image.normalizedDifference(['nir', 'red'])
      lai = ndvi.multiply(2.33).exp().multiply(0.57).rename('LAI')
    elif BaseVI == "EVI":
      # --- Define EVI Constants ---
      G = ee.Number(2.5)
      C1 = ee.Number(6.0)
      C2 = ee.Number(7.5)
      # L = ee.Number(1.0)      #TIP:assumes that the reflectance values are between 0 and 1.
      L = ee.Number(10000)  #TIP:assumes that the reflectance values are between 0 and 10000, common for Sentinel-2 SR and Planet SR in GEE.

      # --- Define LAI Constants for Boegh et al. (2002) ---
      LAI_a = ee.Number(3.618)
      LAI_b = ee.Number(0.118)

      # 1. Calculate EVI (Standard Sentinel-2 EVI formula)
      # EVI = 2.5 * (nir - red) / (nir + 6.0*red - 7.5*blue + 1.0)
      # Get bands
      NIR = image.select('nir')
      RED = image.select('red')
      BLUE = image.select('blue')

      # Calculate numerator and denominator
      numerator = NIR.subtract(RED).multiply(G)
      denominator = NIR.add(RED.multiply(C1)).subtract(BLUE.multiply(C2)).add(L)

      # Compute EVI
      evi = numerator.divide(denominator).rename('EVI')

      # 2. Calculate LAI from EVI (Boegh et al., 2002)
      # Formula: LAI = (3.618 * EVI - 0.118)
      lai = evi.multiply(LAI_a).subtract(LAI_b).rename('LAI')

    else:
        # Handle invalid input
        raise ValueError("BaseVI must be 'NDVI' or 'EVI'.")

    # Ensure LAI is not negative (physically impossible)
    lai = lai.max(0)

    return image.addBands(lai)

# Define the Monthly Iteration Function
def monthly_reduction(year_month):
    """
    Filters the collection for a specific month, calculates the median/mean LAI,
    and adds a time band for later analysis.
    """
    # Parse the year_month string (e.g., '2023-01')
    start = ee.Date(year_month)
    end = start.advance(1, 'month')

    # Filter the full collection for the current month
    monthly_images = full_collection.filterDate(start, end)

    # Check if there are any images in the month
    count = monthly_images.size()

    # Calculate the median LAI for the month
    if BaseStats == "Median":
       monthly_value = monthly_images.median()
    elif BaseStats == "Mean":
       monthly_value = monthly_images.mean()

    # Apply a mask based on the image count: use the median/mean if count > 0; Otherwise, return a masked image (e.g., zero, but masked)
    # result_image = ee.Algorithms.If(count.gt(0), monthly_value, ee.Image(0).selfMask()) #Band name is 'constant'
    dummy = ee.Image(full_collection.first()).multiply(0).selfMask()
    result_image = ee.Image(ee.Algorithms.If(count.gt(0), monthly_value, dummy)) #Band name is 'LAI'

    # Cast the result back to an Image and add properties
    return ee.Image(result_image).clip(roi).set({
        'system:time_start': start.millis(),
        'date_str': start.format('YYYY-MM-dd'),
        'count': count # Useful for tracking data quality
    })


def set_date_from_filename(image):
    # 1. Get the 'system:index' (this is usually the filename in GEE assets)
    # Example: '20240315_153022_94...'
    img_id = image.id()

    # 2. Extract Year, Month, Day (adjust indices based on your filename)
    # For 'YYYYMMDD_...', YYYY is index 0-4, MM is 4-6, DD is 6-8
    year = img_id.slice(0, 4)
    month = img_id.slice(4, 6)
    day = img_id.slice(6, 8)

    # 3. Construct a date string: '2024-03-15'
    date_str = year.cat('-').cat(month).cat('-').cat(day)

    # 4. Convert string to GEE Date and set as 'system:time_start'
    # This property is what GEE uses for all its date filtering/sorting
    return image.set({
        'system:time_start': ee.Date(date_str).millis(),
        'date_str': date_str # Keeping it as a string for your CSV/Table outputs
    })


#----------------------------------------------- MAIN -----------------------------------------------#
### Fetch Planet Data (March 2026)
full_collection = (ee.ImageCollection('projects/quick-heaven-445214-f5/assets/Planet_External')
    .map(set_date_from_filename)
    .filterBounds(roi)
    .filterDate(start_date, end_date)
    .map(calculate_lai)
    .select('LAI')) #LAI for each image
# remove bad_date
full_collection = (
    full_collection
    .filterDate(start_date, bad_date)
    .merge(full_collection.filterDate(ee.Date(bad_date).advance(1, 'day'), end_date))
)


### Generate a list of all month-start dates (YYYY-MM)
start = ee.Date(start_date)
end = ee.Date(end_date).advance(1, 'day')
date_list = ee.List.sequence(0, end.difference(start, 'month').subtract(1)).map(
    lambda n: start.advance(n, 'month')
)
dates = date_list.map(lambda date: ee.Date(date).format('YYYY-MM'))
img_s2_monthly = ee.ImageCollection(dates.map(monthly_reduction))


#----------------------------------------------- MAP -----------------------------------------------#
if False:
    ### find an valid img_s2 that can be shown on Map
    def add_valid_count(img):
        count = img.reduceRegion(
            reducer=ee.Reducer.count(),
            geometry=roi,
            scale=3,
            maxPixels=1e9
        ).get('LAI')
        return img.set('valid_pixel_count', count)

    # 2. Filter the collection to only images that actually HAVE pixels in your ROI
    filtered_for_map = (img_s2_monthly
                        .map(add_valid_count)
                        .filter(ee.Filter.gt('valid_pixel_count', 0))
                        .sort('system:time_start')) # Ensures we get the earliest valid one

    # 3. Select the best image to display
    img_s2 = filtered_for_map.first()
    img_s2_date_val = img_s2.get('date_str').getInfo()
    print(f"img_s2_date: {img_s2_date_val}")

    ### Show LAI_image on google satellite map
    # Check value range
    stats = img_s2.reduceRegion(
        reducer=ee.Reducer.minMax(),
        geometry=roi,
        scale=3
    ).getInfo()
    min_val = stats.get('LAI_min')
    max_val = stats.get('LAI_max')
    print(f"LAI Range for this image: {min_val} to {max_val}")


    # 3. Handle the Map display
    roi_outline = ee.Image().paint(
        featureCollection=ee.FeatureCollection([roi]),
        color=1,
        width=3
    )
    lai_vis = {
        'min': 0,
        'max': 4,
        'palette': ['#f7fcf5', '#c7e9c0', '#41ab5d', '#005a32']
    }
    if min_val is not None:
        Map.centerObject(roi, 15) # Zoom level 15 is good for Planet 3m data
        Map.addLayer(roi_outline, {'palette': 'FF0000'}, 'ROI Boundary')
        Map.addLayer(img_s2, lai_vis, f'Planet LAI ({BaseStats})')
        Map.add_colorbar(
            vis_params=lai_vis,
            label='LAI Value',
            position='bottomleft',
            orientation='horizontal'
        )
    else:
        print("Warning: Image exists but contains no valid pixels in the ROI.")

    Map


In [ ]:
### Show Land Cover on google satellite map
if False:
    # Print Pixel size
    raw_scale = img_lc.projection().nominalScale()
    print('Pixel size (nominalScale):', raw_scale.getInfo())

    # Direct color assignment per value (Visualization)
    class11 = img_lc.eq(11).selfMask().visualize(palette='blue')   # Water
    class24 = img_lc.eq(24).selfMask().visualize(palette='pink')   # Impervious
    class31 = img_lc.eq(31).selfMask().visualize(palette='gray')   # Barren
    class43 = img_lc.eq(43).selfMask().visualize(palette='green')  # Tree
    class71 = img_lc.eq(71).selfMask().visualize(palette='yellow') # Grass

    # Combine by adding layers in order (Blending)
    combined = (class11
                .blend(class24)
                .blend(class31)
                .blend(class43)
                .blend(class71))
    print('Unique values in combined:', combined.getInfo())

    # Show land cover result
    Map.addLayer(combined, {}, 'Styled LC')

    # Display the map
    # Map


Pixel size (nominalScale): 1.7316999503270338
Unique values in combined: {'type': 'Image', 'bands': [{'id': 'vis-red', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'dimensions': [59, 59], 'crs': 'EPSG:4326', 'crs_transform': [1.5556125328877906e-05, 0, -83.90522130789496, 0, -1.5556125328877906e-05, 35.972097753499035]}, {'id': 'vis-green', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'dimensions': [59, 59], 'crs': 'EPSG:4326', 'crs_transform': [1.5556125328877906e-05, 0, -83.90522130789496, 0, -1.5556125328877906e-05, 35.972097753499035]}, {'id': 'vis-blue', 'data_type': {'type': 'PixelType', 'precision': 'int', 'min': 0, 'max': 255}, 'dimensions': [59, 59], 'crs': 'EPSG:4326', 'crs_transform': [1.5556125328877906e-05, 0, -83.90522130789496, 0, -1.5556125328877906e-05, 35.972097753499035]}], 'properties': {'system:annotations': None}}


In [ ]:
# @title Fraction Image
# Get the target resolution (Planet is typically 3 m)
s2_proj = full_collection.first().select(['LAI']).projection()
img_s2_monthly = img_s2_monthly.map(
    lambda img: img.reproject(s2_proj)
)
target_scale = s2_proj.nominalScale().getInfo()
print(f"Target Planet Resolution (Scale): {target_scale} meters")


# Get a list of class values
class_values = list(class_map.keys())

# --- Function to calculate the fractional cover of a single class ---
def calculate_fraction_band(lc_image, class_val):
    """
    Creates a binary mask for a class and calculates its fraction
    at the coarser resolution using ee.Reducer.mean().
    """
    # 1. Create a binary mask: 1 where the pixel equals the class_val, 0 elsewhere
    class_mask = lc_image.eq(class_val)

    # 2. Reduce the resolution to the Planet scale (e.g., 3m)
    # The mean of a binary mask over a region is the fractional area (0.0 to 1.0)
    fraction_band = class_mask.reduceResolution(
        reducer=ee.Reducer.mean(),
        maxPixels=1024  # Standard maxPixels setting
    ).reproject(
        # Reproject to match the Planet image's scale and projection
        crs=s2_proj,
        scale=target_scale
    ).rename(f'Fraction_{class_map[class_val]}')

    return fraction_band

# --- Loop through all classes and combine the fraction bands ---
fraction_bands = []
for val in class_values:
    fraction_band = calculate_fraction_band(img_lc, val)
    fraction_bands.append(fraction_band)

# Combine all fraction bands into a single image
img_fractions = ee.Image(fraction_bands).clip(roi)


### Show fraction on map
frac_vis = {
    'min': 0,
    'max': 1,
    'palette': ['white', 'yellow', 'orange', 'red']
}
for val in class_values:
    class_name = class_map[val]
    band_name = f'Fraction_{class_name}'
    Map.addLayer(
        img_fractions.select(band_name),
        frac_vis,
        f'Fraction: {class_name}'
    )
# Add a shared fraction colorbar legend
Map.add_colorbar(
    vis_params=frac_vis,
    label='Land Cover Fraction (0–1)',
    position='bottomleft',
    orientation='horizontal'
)
# Map


Target Planet Resolution (Scale): 3 meters


In [ ]:
# @title Convert fraction to df
# Sample a Single Monthly Image
def sample_and_format_month(image):
    """
    Combines the monthly LAI image with static fractions, samples the data
    at the ROI, and adds the date as a property to the resulting FeatureCollection.
    """

    # 1. Combine the LAI band (from the current monthly image) with the static Fraction bands
    img_unmixing = image.select('LAI').addBands(img_fractions)

    # 2. Extract data for the ROI
    # Use the projection from the LAI image to ensure consistency
    data_sample = img_unmixing.sample(
        region=roi,
        scale=target_scale,
        projection=image.projection(),
        geometries=False # No need for geometries since we don't use them in the final DF
    )

    # 3. Add the date property to the resulting FeatureCollection
    # The date was stored as 'date_str' during the monthly reduction step
    date_str = image.get('date_str')

    # Map over the sampled features to add the date property to each one
    def add_date(feature):
        return feature.set({'Date': date_str})

    return data_sample.map(add_date)


# ------------------------------------ MAIN ------------------------------------#
print("Extracting time series data for Unmixing...")

# monthly_lai_collection = img_s2_monthly
monthly_lai_collection = img_s2_monthly.filter(
    ee.Filter.inList('date_str', ['2023-08-01'])
)
processed_size = monthly_lai_collection.size().getInfo()
print(f"Number of images in collection: {processed_size}")


# 1. Apply the sampling function over the entire monthly collection
list_of_feature_collections = monthly_lai_collection.map(sample_and_format_month)

# 2. Flatten the List of FeatureCollections into a single FeatureCollection
# This merges all monthly data into one single collection of features
all_data = ee.FeatureCollection(list_of_feature_collections).flatten()

# 3. Convert the large Earth Engine FeatureCollection to a Pandas DataFrame
df = geemap.ee_to_df(all_data)

# --- Final Output ---
print(f"\n--- Extracted Time Series Data ---")
print(f"Total rows (pixels * months): {len(df)}")

# Reorder columns to place 'Date' first and select only relevant bands
band_names = ['LAI'] + [f'Fraction_{name}' for name in class_map.values()]
cols_to_keep = ['Date'] + band_names
df = df[cols_to_keep]

# Display sample of the final DataFrame
print(df.head())
print("\n")
# You can now proceed to run your calculate_lai_per_class function on this 'df'


In [ ]:
# @title Linear Unmixing: to get LC-based LAI
# Note: The 'calculate_lai_per_class' function is the only different code between constrained and unconstrained LAI.
import pandas as pd
import statsmodels.api as sm
from scipy.optimize import nnls
import numpy as np

# --- 1. Your Existing Unmixing Function ---
import numpy as np
import pandas as pd
from scipy.optimize import nnls
from scipy.stats import pearsonr

def calculate_lai_per_class(df_month):
    """
    Calculates non-negative LAI for specific classes,
    fixing Water, Impervious, and Barren to zero.

    Diagnostics:
      - Pseudo R-squared (NNLS, no intercept)
      - Pearson correlation r between observed and fitted LAI
    """

    # 1. Define class groups
    active_classes = ['Fraction_Tree', 'Fraction_Grass']
    zero_classes = ['Fraction_Water', 'Fraction_Impervious', 'Fraction_Barren']

    # 2. Prepare data
    if not all(col in df_month.columns for col in active_classes):
        return None

    Y = df_month['LAI'].values
    X = df_month[active_classes].values

    # Check for sufficient data
    if len(df_month) < len(active_classes):
        return pd.DataFrame({f"LAI_{c.split('_')[1]}": [np.nan] for c in active_classes})

    # 3. Non-negative least squares
    coefs, resnorm = nnls(X, Y)

    # 4. Fitted values
    Y_hat = X @ coefs

    # 5. Extract LAI per class
    flat_data = {}

    for i, col in enumerate(active_classes):
        class_name = col.replace('Fraction_', '')
        flat_data[f"LAI_{class_name}"] = round(coefs[i], 4)

    for col in zero_classes:
        class_name = col.replace('Fraction_', '')
        flat_data[f"LAI_{class_name}"] = 0.0

    # 6. Pseudo R-squared (diagnostic)
    ss_res = np.sum((Y - Y_hat) ** 2)
    ss_tot = np.sum((Y - np.mean(Y)) ** 2)
    pseudo_r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan

    # 7. Pearson correlation r (diagnostic)
    if np.std(Y) > 0 and np.std(Y_hat) > 0 and len(Y) > 2:
        r, _ = pearsonr(Y, Y_hat)
    else:
        r = np.nan

    #flat_data["Pseudo_R2"] = round(pseudo_r2, 4) if not np.isnan(pseudo_r2) else np.nan
    flat_data["Pearson_r"] = round(r, 4) if not np.isnan(r) else np.nan

    return pd.DataFrame([flat_data])



# ----------------- EXECUTION FOR TIME SERIES -----------------
# Assuming the 'df' DataFrame (containing 'Date', 'LAI', and 'Fraction_' columns)
# and has been loaded from the previous Earth Engine extraction step.

if 'Date' not in df.columns:
    print("Error: DataFrame must contain a 'Date' column for time series grouping.")
else:
    print(f"Starting time series unmixing for {df['Date'].nunique()} unique dates...")

    # Group the DataFrame by the 'Date' column
    grouped = df.groupby('Date')

    # Initialize a list to store the results from each month
    all_monthly_results = []

    # Loop through each group (each month)
    for date, df_month in grouped:
        print(f"Processing Date: {date} ({len(df_month)} pixels)")

        # Run the unmixing model on the current month's data
        result_df = calculate_lai_per_class(df_month)

        if result_df is not None and not result_df.empty:
            # The result is a single-row DataFrame, so setting its index to the date works.
            result_df.index = [date]
            all_monthly_results.append(result_df)

    # Concatenate all monthly results into one final DataFrame
    if all_monthly_results:
        LAI_time_series = pd.concat(all_monthly_results)

        # print("\n=== ✅ Final Time Series LAI Endmember Estimates ===")
        # print(LAI_time_series.head())
        # print(f"\nDataFrame shape: {LAI_time_series.shape}")
    else:
        print("\nNo results were generated. Check for data quality or small sample size errors.")


# Save LAI_time_series to CSV
output_filename = f'Planet_LAI_10yrs_{BaseVI}_{BaseStats}_{Site}.csv'
LAI_time_series.to_csv(output_filename)
print(f"DataFrame saved to {output_filename}")

# Download the file to your local computer
from google.colab import files
files.download(output_filename)
print("\n🎉 Download initiated! Check your browser's download folder.")


In [ ]:
# @title *********

In [ ]:
#@title check LAI range across img_s2_monthly
if False:
    valid_monthly = img_s2_monthly

    # 2. Create a "Max" and "Min" composite
    # This finds the highest and lowest LAI value each pixel ever reached
    max_pixel_img = valid_monthly.select('LAI').max()
    min_pixel_img = valid_monthly.select('LAI').min()

    # 3. Calculate the stats over your ROI
    stats_max = max_pixel_img.reduceRegion(
        reducer=ee.Reducer.max(),
        geometry=roi,
        scale=3,  # 10m for Sentinel-2, 3m for Planet
        maxPixels=1e9
    ).getInfo()

    stats_min = min_pixel_img.reduceRegion(
        reducer=ee.Reducer.min(),
        geometry=roi,
        scale=3,
        maxPixels=1e9
    ).getInfo()

    print(f"Actual Pixel LAI Min: {stats_min.get('LAI')}")
    print(f"Actual Pixel LAI Max: {stats_max.get('LAI')}")

In [ ]:
# @title Check missing months
# Missing months: May and Nov, 2024
# Result: there were no Planet images available that met the cloud cover criteria (less than 20%))
# Assuming img_s2_monthly is available from previous cells

if True:
  # Get a list of dictionaries containing 'date_str' and 'count' for each image in the collection
  counts_list = img_s2_monthly.aggregate_array('count').getInfo()
  date_strs_list = img_s2_monthly.aggregate_array('date_str').getInfo()

  print("Monthly Image Counts after Cloud Filtering:")
  for i in range(len(counts_list)):
      print(f"Date: {date_strs_list[i]}, Images Count: {counts_list[i]}")


In [ ]:
# @title Check LAI time series
import matplotlib.pyplot as plt
import pandas as pd

# Ensure the index is in datetime format for proper plotting
LAI_time_series.index = pd.to_datetime(LAI_time_series.index)

# Subset for 2023-2024
LAI_time_series_subset = LAI_time_series.loc['2023':'2024']

# Select only the LAI columns for plotting
lai_columns = [col for col in LAI_time_series_subset.columns if col.startswith('LAI_')]
plot_data = LAI_time_series_subset[lai_columns]

# Create the plot
plt.figure(figsize=(14, 7))

for column in plot_data.columns:
    plt.plot(plot_data.index, plot_data[column], marker='o', linestyle='-', label=column.replace('LAI_', ''))

plt.title(f'Estimated LAI for Each Land Cover Class Over Time (2023-2024) at {Site}')
plt.xlabel('Date')
plt.ylabel('Estimated LAI')
plt.ylim(-5,5)
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend(title='Land Cover Class')
plt.tight_layout()
# plt.show()


### Save and Download the Image ---
image_filename = f'Planet_LAI_2yrs_{BaseVI}_{BaseStats}_{Site}.png'
plt.savefig(image_filename, dpi=300, bbox_inches='tight')
files.download(image_filename)
print("🎉 Download initiated! Check your browser's download folder.")

